In [ ]:
%pip install -r requirements.txt

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv("/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/documents.csv", index_col ="document_id")
test = pd.read_csv("/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/test_queries.csv")
df.head()
test.head()

In [ ]:
# My plan is to use hybird method which involves
# bm25 + dense retrival + reranker
def create_search_content(row):
    title = str(row.get("title", " ")).strip()
    text = str(row.get("text", " ")).strip()
    source = str(row.get("source", " ")).strip()
    crop = str(row.get("crop", " ")).strip()
    country = str(row.get("country", " ")).strip()
    source_url = str(row.get("source_url", " ")).strip()
    
    return f"Crop {crop} | Country {country} | Title {title} | Text {text} | Source {source}"
df_upd=pd.DataFrame()
df["search_text"] = df.apply(create_search_content, axis = 1)
docs_ids = df.index.to_list()
corpus_texts = df['search_text'].tolist()

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

# ----------------------------------------------------
# 1. Multi-GPU Device Setup & Model Initialization
# ----------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_gpus = torch.cuda.device_count()
print(f"Using device: {device} | Active GPUs: {num_gpus}")

model_name = "BAAI/bge-reranker-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)

# Move model to device and wrap in DataParallel if multiple GPUs exist
model.to(device)
if num_gpus > 1:
    print(f"Distributing across {num_gpus} GPUs via DataParallel.")
    model = nn.DataParallel(model)

# ----------------------------------------------------
# 2. Build Training Pairs (Positives + Hard Negatives)
# ----------------------------------------------------
train_queries_df = pd.read_csv('/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/train_queries.csv')
qrels_train_df = pd.read_csv('/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/qrels_train.csv')

tq_id_col = 'QueryId' if 'QueryId' in train_queries_df.columns else train_queries_df.columns[0]
tq_text_col = 'Query' if 'Query' in train_queries_df.columns else train_queries_df.columns[1]

qrels_qid_col = 'QueryId' if 'QueryId' in qrels_train_df.columns else qrels_train_df.columns[0]
qrels_did_col = 'DocumentId' if 'DocumentId' in qrels_train_df.columns else qrels_train_df.columns[1]
query_dict = dict(zip(train_queries_df[tq_id_col], train_queries_df[tq_text_col]))
doc_dict = dict(zip(docs_ids, corpus_texts))
# Keep the relevance scores
qrels = qrels_train_df.copy()

# Group documents by query
qrels_by_query = qrels.groupby(qrels_qid_col)

train_queries = []
train_docs_a = []
train_docs_b = []

for q_id, group in qrels_by_query:
    if q_id not in query_dict:
        continue

    q_text = str(query_dict[q_id])

    # Compare every pair where relevance differs
    rows = group.to_dict("records")

    for a in rows:
        for b in rows:

            score_a = float(a["relevance"])
            score_b = float(b["relevance"])

            # A must be more relevant than B
            if score_a > score_b:
                doc_a = a[qrels_did_col]
                doc_b = b[qrels_did_col]

                if doc_a in doc_dict and doc_b in doc_dict:
                    train_queries.append(q_text)
                    train_docs_a.append(doc_dict[doc_a])
                    train_docs_b.append(doc_dict[doc_b])

print(f"Training ranking pairs: {len(train_queries):,}")
# ----------------------------------------------------
# 3. Dataset & Multi-GPU DataLoader
# ----------------------------------------------------
class PairDataset(Dataset):
    def __init__(self, queries, docs_a, docs_b, tokenizer, max_length=256):
        self.queries = queries
        self.docs_a = docs_a
        self.docs_b = docs_b
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.queries)

    def __getitem__(self, idx):
        inputs_a = self.tokenizer(
            self.queries[idx],
            self.docs_a[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        inputs_b = self.tokenizer(
            self.queries[idx],
            self.docs_b[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids_a": inputs_a["input_ids"].squeeze(0),
            "attention_mask_a": inputs_a["attention_mask"].squeeze(0),
            "input_ids_b": inputs_b["input_ids"].squeeze(0),
            "attention_mask_b": inputs_b["attention_mask"].squeeze(0),
        }

train_dataset = PairDataset(
    train_queries,
    train_docs_a,
    train_docs_b,
    tokenizer
)

# Scale batch size by number of GPUs (4 per GPU)
effective_batch_size = 2 * max(1, num_gpus)
train_loader = DataLoader(train_dataset, batch_size=effective_batch_size, shuffle=True)

# ----------------------------------------------------
# 4. Multi-GPU Training Loop
# ----------------------------------------------------
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
criterion = nn.MarginRankingLoss(margin=0.2)
epochs = 4

total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps
)

model.train()
for epoch in range(epochs):
    running_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
    
        input_ids_a = batch["input_ids_a"].to(device)
        attention_mask_a = batch["attention_mask_a"].to(device)
    
        input_ids_b = batch["input_ids_b"].to(device)
        attention_mask_b = batch["attention_mask_b"].to(device)
    
        outputs_a = model(
            input_ids=input_ids_a,
            attention_mask=attention_mask_a
        )
    
        outputs_b = model(
            input_ids=input_ids_b,
            attention_mask=attention_mask_b
        )
    
        score_a = outputs_a.logits.squeeze(-1)
        score_b = outputs_b.logits.squeeze(-1)
    
        # Tell the loss: A should score higher than B
        target = torch.ones_like(score_a)
    
        loss = criterion(score_a, score_b, target)
    
        loss.backward()
    
        optimizer.step()
        scheduler.step()
    
        running_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss / len(train_loader):.4f}")

# ----------------------------------------------------
# 5. Save the Fine-Tuned Model to Kaggle Disk
# ----------------------------------------------------
save_dir = "./fine_tuned_bge_reranker"
os.makedirs(save_dir, exist_ok=True)

# Unwrap DataParallel before saving to retain standard model format
model_to_save = model.module if isinstance(model, nn.DataParallel) else model

model_to_save.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"Fine-tuning complete! Model saved successfully to '{save_dir}'.")